# Archive Metadata Summary

Quick notebook to inspect the current archive footprint:
- total PDFs on disk
- total documents in `archive.db`
- total chunks in `chunks.db`
- rough chunk size stats
- a few useful coverage summaries


In [ ]:
from pathlib import Path
import sqlite3
from statistics import mean, median

PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "data_pipeline":
    PIPELINE_DIR = PIPELINE_DIR / "data_pipeline"

PROJECT_ROOT = PIPELINE_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / "output"
PDF_DIR = OUTPUT_DIR / "pdfs"
ARCHIVE_DB = OUTPUT_DIR / "metadata" / "archive.db"
CHUNKS_DB = OUTPUT_DIR / "metadata" / "chunks.db"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_DIR exists:", PDF_DIR.exists())
print("ARCHIVE_DB exists:", ARCHIVE_DB.exists())
print("CHUNKS_DB exists:", CHUNKS_DB.exists())


In [ ]:
pdf_paths = list(PDF_DIR.rglob("*.pdf"))
pdf_count = len(pdf_paths)
pdf_total_bytes = sum(path.stat().st_size for path in pdf_paths if path.exists())
pdf_empty_count = sum(1 for path in pdf_paths if path.exists() and path.stat().st_size == 0)

print("Total PDFs on disk:", pdf_count)
print("Total PDF size (GB):", round(pdf_total_bytes / (1024 ** 3), 2))
print("Empty PDF files:", pdf_empty_count)


In [ ]:
archive_conn = sqlite3.connect(ARCHIVE_DB)
archive_conn.row_factory = sqlite3.Row
chunks_conn = sqlite3.connect(CHUNKS_DB)
chunks_conn.row_factory = sqlite3.Row

document_count = archive_conn.execute("SELECT COUNT(*) AS n FROM documents").fetchone()["n"]
chunk_count = chunks_conn.execute("SELECT COUNT(*) AS n FROM chunks").fetchone()["n"]

print("Documents in archive.db:", document_count)
print("Chunks in chunks.db:", chunk_count)
print("Average chunks per document:", round(chunk_count / document_count, 2) if document_count else None)


In [ ]:
chunk_stats = chunks_conn.execute(
    """
    SELECT
        AVG(word_count) AS avg_words,
        MIN(word_count) AS min_words,
        MAX(word_count) AS max_words,
        AVG(LENGTH(text)) AS avg_chars,
        MIN(LENGTH(text)) AS min_chars,
        MAX(LENGTH(text)) AS max_chars
    FROM chunks
    """
).fetchone()

print("Average chunk size (words):", round(chunk_stats["avg_words"], 1) if chunk_stats["avg_words"] is not None else None)
print("Chunk size range (words):", chunk_stats["min_words"], "to", chunk_stats["max_words"])
print("Average chunk size (chars):", round(chunk_stats["avg_chars"], 1) if chunk_stats["avg_chars"] is not None else None)
print("Chunk size range (chars):", chunk_stats["min_chars"], "to", chunk_stats["max_chars"])


In [ ]:
distribution_rows = chunks_conn.execute(
    "SELECT word_count FROM chunks ORDER BY word_count"
).fetchall()
word_counts = [row[0] for row in distribution_rows]

def percentile(values, p):
    if not values:
        return None
    idx = min(len(values) - 1, max(0, round((len(values) - 1) * p)))
    return values[idx]

print("Median chunk size (words):", median(word_counts) if word_counts else None)
print("10th / 90th percentile (words):", percentile(word_counts, 0.10), "/", percentile(word_counts, 0.90))


In [ ]:
coverage = chunks_conn.execute(
    """
    SELECT
        COUNT(*) AS total_chunks,
        SUM(CASE WHEN page_number IS NOT NULL THEN 1 ELSE 0 END) AS mapped_chunks,
        AVG(CASE WHEN page_match_score IS NOT NULL THEN page_match_score END) AS avg_page_match_score
    FROM chunks
    """
).fetchone()

mapped_chunks = coverage["mapped_chunks"] or 0
total_chunks = coverage["total_chunks"] or 0

print("Chunks with page mapping:", mapped_chunks)
print("Page mapping coverage:", round(100 * mapped_chunks / total_chunks, 2) if total_chunks else None, "%")
print("Average page match score:", round(coverage["avg_page_match_score"], 4) if coverage["avg_page_match_score"] is not None else None)


In [ ]:
top_years = archive_conn.execute(
    """
    SELECT year, COUNT(*) AS document_count
    FROM documents
    GROUP BY year
    ORDER BY document_count DESC, year
    LIMIT 10
    """
).fetchall()

for row in top_years:
    print(dict(row))


In [ ]:
sample_documents = archive_conn.execute(
    """
    SELECT doc_id, date, title
    FROM documents
    ORDER BY date
    LIMIT 5
    """
).fetchall()

for row in sample_documents:
    print(dict(row))

archive_conn.close()
chunks_conn.close()
